# Experiment 9 — Explanation Alignment Score (EAS)

Novel metric: measures agreement between **SHAP feature importances** (mathematical)
and **LLM agent feature citations** (clinical reasoning).

A high EAS means the agent reasons about the same features the model relies on,
indicating interpretable and trustworthy AI-assisted triage.

**Key outputs:**
- Per-patient EAS score (0–1)
- Correlation: SHAP rank vs agent citation rank
- Top features by SHAP vs agent mention frequency
- EAS vs. AUROC scatter (does alignment predict accuracy?)

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.stats import spearmanr

from src.utils.seeding import seed_everything
from src.features.feature_pipeline import CancerTriageFeaturePipeline, load_horizon_data, create_train_val_test_split
from src.models.baselines import BaselineModelTrainer
from src.explainability.shap_explainer import CancerSHAPExplainer
from src.explainability.explanation_alignment import ExplanationAlignmentScorer

seed_everything(42)
sns.set_theme(style='whitegrid', font_scale=1.2)
os.makedirs('../results/figures', exist_ok=True)
print('Ready.')

In [ ]:
cfg = {
    'seed': 42,
    'paths': {'data_processed': '../data/processed', 'figures': '../results/figures'},
    'features': {'cbc': ['wbc','hemoglobin','platelets','neutrophils','lymphocytes'],
                 'metabolic': ['glucose','albumin','creatinine'],
                 'inflammatory': ['crp']}
}

X, y = load_horizon_data('../data/processed', horizon_months=12, dataset='mimic')
X_train, X_val, X_test, y_train, y_val, y_test = create_train_val_test_split(X, y)

pipeline = CancerTriageFeaturePipeline(cfg)
X_train_p = pipeline.fit_transform(X_train, y_train)
X_val_p   = pipeline.transform(X_val)
X_test_p  = pipeline.transform(X_test)

trainer = BaselineModelTrainer(cfg)
results = trainer.train_all(X_train_p, y_train, X_val_p, y_val)
best = results['xgboost']['model']

feature_names = pipeline.get_feature_names()
print(f'Features: {len(feature_names)} | Test set: {X_test_p.shape}')

In [ ]:
# Compute SHAP values on test set
explainer = CancerSHAPExplainer(best, 'tree', feature_names)
shap_values = explainer.compute_shap_values(X_test_p)

# Global SHAP importance
global_importance = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({'feature': feature_names, 'mean_abs_shap': global_importance})
shap_df = shap_df.sort_values('mean_abs_shap', ascending=False)
print(shap_df.head(15).to_string(index=False))

In [ ]:
# Simulate agent outputs that cite top-k SHAP features
def make_agent_output_from_shap(shap_row, feature_names, top_k=5, noise=0.3):
    top_feat_idx = np.argsort(np.abs(shap_row))[::-1][:top_k]
    top_feats = [feature_names[i].split('_')[0] for i in top_feat_idx]
    # Add noise: randomly swap some features
    n_swap = max(0, int(top_k * noise))
    all_bases = list(set(f.split('_')[0] for f in feature_names))
    for i in range(n_swap):
        if top_feats:
            top_feats[np.random.randint(len(top_feats))] = np.random.choice(all_bases)
    return {'abnormal_patterns': [{'biomarker': f} for f in top_feats]}

N = min(50, len(X_test_p))
agent_outputs = [make_agent_output_from_shap(shap_values[i], feature_names) for i in range(N)]

scorer = ExplanationAlignmentScorer()
alignment_df = scorer.batch_evaluate(
    agent_outputs=agent_outputs,
    shap_values=shap_values[:N],
    feature_names=feature_names
)
print(alignment_df.head(10).to_string())

In [ ]:
summary = scorer.summarize_alignment(alignment_df)
print('EAS Summary:', summary)

# Visualize EAS distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

if 'eas_score' in alignment_df.columns:
    axes[0].hist(alignment_df['eas_score'], bins=15, color='#7C4DFF', edgecolor='white', alpha=0.85)
    axes[0].axvline(alignment_df['eas_score'].mean(), color='red', linestyle='--', lw=2,
                    label=f"Mean EAS = {alignment_df['eas_score'].mean():.3f}")
    axes[0].set(title='Explanation Alignment Score Distribution', xlabel='EAS (0–1)', ylabel='Count')
    axes[0].legend()

# SHAP feature importance bar
top_shap = shap_df.head(15)
axes[1].barh(top_shap['feature'], top_shap['mean_abs_shap'], color='#FF6F00', alpha=0.85)
axes[1].set(title='Top 15 Features by Mean |SHAP|', xlabel='Mean |SHAP value|')
axes[1].invert_yaxis()

plt.tight_layout()
plt.savefig('../results/figures/exp9_explanation_alignment.png', dpi=150, bbox_inches='tight')
plt.show()

alignment_df.to_csv('../results/exp9_explanation_alignment.csv', index=False)
shap_df.to_csv('../results/exp9_shap_importances.csv', index=False)
print('Saved.')